<a href="https://colab.research.google.com/github/ibtihal7alharbi-tech/Computer-vision-project-/blob/main/Computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Ultralytics (YOLOv8)
!pip install ultralytics --quiet

# OpenCV for image processing
!pip install opencv-python --quiet

# Matplotlib for visualization
!pip install matplotlib --quiet

# (Optional but useful)
!pip install seaborn pandas --quiet


!pip install roboflow --quiet


!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-python


Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.10.0.84
Uninstalling opencv-python-headless-4.10.0.84:
  Successfully uninstalled opencv-python-headless-4.10.0.84
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)


In [2]:
!pip install roboflow

  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.9 MB)


In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key="ZeJKDEWv7ZknZBJqtk1s")
project = rf.workspace("ibtihals-workspace").project("sids-mdtwn")
version = project.version(2)
dataset = version.download("yolov8")

from ultralytics import YOLO
import cv2
import os

# Load models
body_model = YOLO("body_best.pt")   # trained on: back, stomach
face_model = YOLO("face_best.pt")   # trained on: head, nose

test_folder = "/content/sids-2/test/images"
output_folder = "results"
os.makedirs(output_folder, exist_ok=True)

results_summary = []

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)
    frame = cv2.imread(img_path)

    # Run models
    # 1.(Run models)
    body_results = body_model(frame, verbose=False)[0]
    face_results = face_model(frame, verbose=False)[0]

# -------- Gather all results in one place --------
    all_boxes = []
# We combine the results of both models so that we don't lose any information
    for res in [body_results, face_results]:
        for box in res.boxes:
            all_boxes.append({
                "label": res.names[int(box.cls[0])],
                "conf": float(box.conf[0])
            })

    # --------(Position) --------
    position = "none"
    best_pos_conf = 0

    # -------- (Face) --------
    head_detected = False
    nose_detected = False

    for item in all_boxes:
        lbl = item["label"].lower()
        cnm = item["conf"]

# We capture the position from any model that has captured it (with a low confidence of 0.2 to ensure capture)
        if ("back" in lbl or "stomach" in lbl) and cnm > 0.2:
            if cnm > best_pos_conf:
                best_pos_conf = cnm
                position = lbl

# We pick up the head and nose
        if ("baby" in lbl or "head" in lbl) and cnm > 0.3:
            head_detected = True
        if "nose" in lbl and cnm > 0.3:
            nose_detected = True

    #--------- logic section---------

    status = "UNKNOWN"
    color = (255, 255, 255)

#1. If the model detects abdominal position -> danger immediately
    if "stomach" in position:
        status = "DANGER: STOMACH SLEEPING"
        color = (0, 0, 255)

#2. If the model detects a back position -> SAFE
    elif "back" in position:
        status = "SAFE"
        color = (0, 255, 0)

# 3. If no clear position is found (or if the value is "none")
    else:
        status = f"UNKNOWN ({position})"
        color = (255, 255, 255)
    # Save results
    results_summary.append({
        "image": img_name,
        "position": position,
        "head": head_detected,
        "nose": nose_detected,
        "status": status
    })

    # -------- DRAW --------
    annotated = frame.copy()

    # Body boxes (blue)
    for box in body_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = f"{body_model.names[int(box.cls[0])]} {float(box.conf[0]):.2f}"
        cv2.rectangle(annotated, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(annotated, label, (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Face boxes (green)
    for box in face_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = f"{face_model.names[int(box.cls[0])]} {float(box.conf[0]):.2f}"
        cv2.rectangle(annotated, (x1,y1), (x2,y2), (255,0,0), 2)
        cv2.putText(annotated, label, (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

    # Status text
    color = (0,255,0) if "SAFE" in status else (0,0,255)
    cv2.putText(annotated, status, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imwrite(os.path.join(output_folder, img_name), annotated)

print("Inference done ")


loading Roboflow workspace...
loading Roboflow project...
Inference done ✅


In [17]:
# trying the model on real videos- to try this model

#1- you have to upload the video into cloab through the file section
#2-the copy the path of the video and paste it in line 15
#3-check after run for a new file called "Baby_Guard_Final_Result.mp4

import cv2
from ultralytics import YOLO
import os

# 1. load models
print("loading models ...")
body_model = YOLO("body_best.pt")
face_model = YOLO("face_best.pt")

#Video setting

video_input = "/content/WhatsApp Video 2026-04-17 at 16.05.31.mp4"
cap = cv2.VideoCapture(video_input)

if not cap.isOpened():
 print("Error: The video file was not found. Please check the path.")
else:
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = "Baby_Guard_Final_Result.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    danger_counter = 0
    WAIT_SECONDS = 10
    frames_threshold = fps * WAIT_SECONDS

    print(f" Processing has begun... The video will be saved as: {output_path}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        body_results = body_model(frame, verbose=False, conf=0.25)[0]
        face_results = face_model(frame, verbose=False, conf=0.25)[0]

        all_labels = []
        nose_found = False
        for res in [body_results, face_results]:
            for box in res.boxes:
                lbl = res.names[int(box.cls[0])].lower()
                all_labels.append(lbl)
                if "nose" in lbl: nose_found = True

        combined_labels = " ".join(all_labels)
# --- (The Smart Priority Logic) ---
        status = "SCANNING..."
        color = (255, 255, 255)

        is_on_back = False
        is_on_stomach = False
        nose_found = False

# We go over all the squares that the two models picked up
        all_labels = []
        for res in [body_results, face_results]:
            for box in res.boxes:
                lbl = res.names[int(box.cls[0])].lower()
                all_labels.append(lbl)
                if "nose" in lbl: nose_found = True

# Merge all labels into one text for easy searching

        combined_labels = " ".join(all_labels)

        if "stomach" in combined_labels:
            danger_counter += 1
            if danger_counter < frames_threshold:
                remaining = WAIT_SECONDS - (danger_counter // fps)
                status = f"WARNING: STOMACH ({remaining}s)"
                color = (0, 165, 255)
            else:
                status = " DANGER: STOMACH "
                color = (0, 0, 255)

        elif "back" in combined_labels:
            danger_counter = 0
            status = " SAFE"
            color = (0, 255, 0)

        elif nose_found:
            danger_counter = 0
            status = "SAFE (Nose Visible)"
            color = (0, 255, 0)

        else:
            status = f"SCANNING... ({combined_labels[:15]})"

        # --- (Bounding Boxes) ---
        annotated_frame = frame.copy()

# plot everything the camera captured
        for res, b_color in [(body_results, (255,0,0)), (face_results, (0,255,0))]:
            for box in res.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                lbl = res.names[int(box.cls[0])]
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), b_color, 2)
                cv2.putText(annotated_frame, lbl, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, b_color, 2)

        cv2.rectangle(annotated_frame, (0, 0), (width, 100), (0,0,0), -1)
        cv2.putText(annotated_frame, status, (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 4)

        out.write(annotated_frame)
    cap.release()
    out.release()

    print(f"Processed successfully! File located here: {output_path}")

loading models ...
 Processing has begun... The video will be saved as: Baby_Guard_Final_Result.mp4
Processed successfully! File located here: Baby_Guard_Final_Result.mp4
